---
# PARTIE 2 -- L'API DataFrame

> **Démarrage :** exécutez la **Section 0** en premier.
> Elle configure Spark et construit `step3` si vous ouvrez ce notebook sans la Session 1.
>
> **Erreur Py4J ?** Kernel → Restart, puis la cellule **Reset Spark**, puis Section 0.

## 2.1 Du RDD au DataFrame

L'API RDD est puissante mais verbeuse et peu optimisee : Spark ne sait pas ce que
contiennent les elements (ils sont des `object` Python opaques). L'API **DataFrame**
introduit un schema explicite, ce qui permet a l'optimiseur Catalyst de generer
un plan d'execution bien plus efficace.

### Creation depuis un RDD


### Reset Spark (si erreur Py4J / RPC)

En cas de `Py4JNetworkError`, `Answer from Java side is empty` ou `jsc is null` :
1. **Kernel → Restart Kernel**
2. Exécutez la cellule ci-dessous
3. Puis relancez la **Section 0**


In [43]:
from pyspark.sql import SparkSession

for _name in ("spark", "sc", "step3"):
    if _name in globals():
        globals()[_name] = None

try:
    from pyspark.sql.context import SQLContext
    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None
except Exception as exc:
    print(f"clearSession ignoré : {exc}")

print("Variables Spark remises à zéro. Relancez la Section 0.")


Variables Spark remises à zéro. Relancez la Section 0.


In [44]:
# ── Section 0 : config + Spark + step3 (autonome ou reprise Session 1) ───────
import csv
import os
import sys
import time
import platform
import subprocess
from pathlib import Path

from pyspark.sql import SparkSession

# Mac Apple Silicon : Java arm64 + workers PySpark arm64
if platform.system() == "Darwin" and platform.machine() == "arm64":
    _jdk_candidates = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
    ]
    _jdk = next((p for p in _jdk_candidates if (p / "bin" / "java").exists()), None)
    if _jdk:
        os.environ["JAVA_HOME"] = str(_jdk)
        os.environ["PATH"] = f"{_jdk / 'bin'}:{os.environ.get('PATH', '')}"

    _python = Path(sys.executable)
    _wrapper = _python.parent / "python-arm64"
    if not _wrapper.exists():
        _wrapper.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{_python}" "$@"\n',
            encoding="utf-8",
        )
        _wrapper.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(_wrapper)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(_python)

    _java_ver = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print(f"[Java] {(_java_ver.stderr or _java_ver.stdout).splitlines()[0]}")

# Chemins et paramètres (alignés Session 1)
_candidats = [Path("data"), Path("../data")]
DATA_DIR = next((p for p in _candidats if (p / "velib").exists()), _candidats[0])
HISTORIQUE_STATIONS_CSV = next(
    (p for p in [
        Path("historique_stations.csv"),
        DATA_DIR / "velib" / "raw" / "historique_stations.csv",
    ] if p.exists()),
    Path("historique_stations.csv"),
)
VELIB_RAW_DIR = DATA_DIR / "velib" / "raw"
VELIB_PARQ_DIR = DATA_DIR / "velib" / "parquet"
STATIONS_CSV = DATA_DIR / "velib" / "stations_info.csv"
METEO_CSV = DATA_DIR / "meteo" / "paris_montsouris_horaire.csv"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

APP_NAME = "ClimaCity-Paris"
MASTER = "local[*]"
SHUFFLE_PARTS = 8

def _clear_spark_session_registry() -> None:
    from pyspark.sql.context import SQLContext

    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None


def _spark_is_alive() -> bool:
    if "spark" not in globals() or spark is None:
        return False
    try:
        _sc = spark.sparkContext
        return _sc is not None and _sc._jsc is not None
    except Exception:
        return False


def _reset_spark_locals() -> None:
    global spark, sc
    spark = None
    sc = None


def _create_spark_session():
    global spark, sc

    if _spark_is_alive():
        try:
            spark.stop()
        except Exception:
            pass
    else:
        _reset_spark_locals()

    _clear_spark_session_registry()

    _reset_spark_locals()

    spark = (
        SparkSession.builder
        .appName(APP_NAME)
        .master(MASTER)
        .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
        .config("spark.driver.memory", "4g")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    sc = spark.sparkContext
    sc.setLogLevel("WARN")
    print(f"[Spark] session créée — version {spark.version}")
    return spark, sc


if not _spark_is_alive():
    spark, sc = _create_spark_session()
else:
    sc = spark.sparkContext
    sc.setLogLevel("WARN")
    print("[Spark] session réutilisée")


def _step3_is_stale() -> bool:
    if "step3" not in globals():
        return True
    try:
        return step3.context is not sc
    except Exception:
        return True


# Pipeline RDD → step3 (même logique que Session 1, exercices 7–8)
if _step3_is_stale():
    if not HISTORIQUE_STATIONS_CSV.exists():
        raise FileNotFoundError(f"Fichier introuvable : {HISTORIQUE_STATIONS_CSV}")

    def parse_ligne(line: str) -> dict | None:
        try:
            champs = next(csv.reader([line]))
            if len(champs) != 7:
                return None
            horodatage, capacite, velos_meca, velos_elec, nom_station, coordonnees, operative = champs
            cap = int(capacite)
            meca = int(velos_meca)
            elec = int(velos_elec)
            return {
                "horodatage": horodatage,
                "capacite": cap,
                "velos_meca": meca,
                "velos_elec": elec,
                "bornettes_libres": cap - meca - elec,
                "nom_station": nom_station,
                "coordonnees": coordonnees,
                "operative": operative.lower() == "true",
            }
        except (ValueError, StopIteration):
            return None

    print(f"[RDD] construction de step3 depuis {HISTORIQUE_STATIONS_CSV.name}…")
    t0 = time.perf_counter()
    raw_rdd = sc.textFile(str(HISTORIQUE_STATIONS_CSV))
    entete = raw_rdd.first()
    data_rdd = raw_rdd.filter(lambda line: line != entete)
    clean_rdd = data_rdd.map(parse_ligne).filter(lambda x: x is not None)
    step1 = clean_rdd.filter(
        lambda r: r["capacite"] > 0 and (r["velos_meca"] + r["velos_elec"]) > 0
    )

    def ajouter_taux(r):
        cap = r["capacite"]
        velos = r["velos_meca"] + r["velos_elec"]
        return {
            **r,
            "velos_total": velos,
            "taux_occupation": (cap - r["bornettes_libres"]) / cap,
        }

    step2 = step1.map(ajouter_taux)
    step3 = step2.filter(lambda r: r["taux_occupation"] < 0.10)
    print(f"[RDD] step3 prêt (plan construit en {(time.perf_counter()-t0)*1000:.0f} ms)")
else:
    print("[RDD] step3 réutilisé (même SparkContext)")

[Java] openjdk version "17.0.19" 2026-04-21
[Spark] session créée — version 3.5.9
[RDD] construction de step3 depuis historique_stations.csv…
[RDD] step3 prêt (plan construit en 177 ms)


In [45]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, BooleanType,
)

# Schéma aligné sur les dicts produits par parse_ligne() + ajouter_taux()
schema_velib = StructType([
    StructField("horodatage",       StringType(),  nullable=True),
    StructField("capacite",         IntegerType(), nullable=True),
    StructField("velos_meca",       IntegerType(), nullable=True),
    StructField("velos_elec",       IntegerType(), nullable=True),
    StructField("bornettes_libres", IntegerType(), nullable=True),
    StructField("nom_station",      StringType(),  nullable=True),
    StructField("coordonnees",      StringType(),  nullable=True),
    StructField("operative",        BooleanType(), nullable=True),
    StructField("velos_total",      IntegerType(), nullable=True),
    StructField("taux_occupation",  DoubleType(),  nullable=True),
])

# Vérifier que step3 appartient bien à la session Spark active
if step3.context is not sc:
    raise RuntimeError(
        "step3 est lié à une ancienne session Spark. Relancez la Section 0 du notebook."
    )

row_rdd = step3.map(lambda d: Row(**d))
df_from_rdd = spark.createDataFrame(row_rdd, schema=schema_velib)

df_from_rdd.printSchema()
df_from_rdd.show(5, truncate=False)
print(f"Lignes : {df_from_rdd.count():,}  --  Partitions : {df_from_rdd.rdd.getNumPartitions()}")


root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- operative: boolean (nullable = true)
 |-- velos_total: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)



26/07/21 14:08:46 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 238 (TID 579): Attempting to kill Python Worker


+-----------------+--------+----------+----------+----------------+---------------------------+----------------+---------+-----------+--------------------+
|horodatage       |capacite|velos_meca|velos_elec|bornettes_libres|nom_station                |coordonnees     |operative|velos_total|taux_occupation     |
+-----------------+--------+----------+----------+----------------+---------------------------+----------------+---------+-----------+--------------------+
|2020-11-26T12:59Z|21      |0         |1         |20              |Toudouze - Clauzel         |48.87930,2.33736|true     |1          |0.047619047619047616|
|2020-11-26T12:59Z|60      |2         |2         |56              |Alibert - Jemmapes         |48.87104,2.36610|true     |4          |0.06666666666666667 |
|2020-11-26T12:59Z|25      |0         |2         |23              |Cassini - Denfert-Rochereau|48.83753,2.33604|true     |2          |0.08                |
|2020-11-26T12:59Z|23      |1         |0         |22            

---
## 2.2 Lecture directe depuis les fichiers Parquet

En pratique, on ne passe presque jamais par les RDD pour creer un DataFrame.
Spark peut lire directement de nombreux formats (CSV, JSON, Parquet, Delta, ORC...).

Le format **Parquet** est le format de facto pour les donnees analytiques :
- Stockage colonnaire (lecture partielle possible)
- Compression integree (Snappy, Zstandard...)
- Schema embarque dans les fichiers
- Partitionnement natif (Hive-style)


In [46]:
from pyspark.sql.functions import to_timestamp, year, month, col

# Lecture des fichiers Parquet partitionnes par annee et mois
# Si le dossier est vide, on materialise df_from_rdd en Parquet (une seule fois)
VELIB_PARQ_DIR.mkdir(parents=True, exist_ok=True)
if not any(VELIB_PARQ_DIR.glob("**/*.parquet")):
    print(f"[Parquet] dossier vide — écriture depuis df_from_rdd → {VELIB_PARQ_DIR}")
    (
        df_from_rdd
        .withColumn("ts", to_timestamp("horodatage", "yyyy-MM-dd'T'HH:mmX"))
        .withColumn("annee", year("ts"))
        .withColumn("mois", month("ts"))
        .drop("ts")
        .write
        .mode("overwrite")
        .partitionBy("annee", "mois")
        .parquet(str(VELIB_PARQ_DIR))
    )

df_velib = spark.read.parquet(str(VELIB_PARQ_DIR))

df_velib.printSchema()
print(f"\nDimensions : {df_velib.count():,} lignes x {len(df_velib.columns)} colonnes")
print(f"Partitions : {df_velib.rdd.getNumPartitions()}")


root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- operative: boolean (nullable = true)
 |-- velos_total: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)


Dimensions : 690,858 lignes x 12 colonnes
Partitions : 8


In [47]:
# Lecture selective d'une partition (predicate pushdown)
# Spark ne lira que les fichiers du dossier annee=2020/mois=11
df_nov_2020 = spark.read.parquet(str(VELIB_PARQ_DIR)).filter("annee = 2020 AND mois = 11")
print(f"Novembre 2020 : {df_nov_2020.count():,} lignes")

# Comparer avec une lecture complete puis filtre — le plan d'execution est identique
# grace au predicate pushdown, mais on le verifie avec explain()
print("\n--- Plan d'execution (filtre apres lecture) ---")
df_velib.filter("annee = 2020 AND mois = 11").explain(mode="formatted")


Novembre 2020 : 69,208 lignes

--- Plan d'execution (filtre apres lecture) ---
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [12]: [horodatage#10824, capacite#10825, velos_meca#10826, velos_elec#10827, bornettes_libres#10828, nom_station#10829, coordonnees#10830, operative#10831, velos_total#10832, taux_occupation#10833, annee#10834, mois#10835]
Batched: true
Location: InMemoryFileIndex [file:/Users/romain/Desktop/SparkVelib/data/velib/parquet]
PartitionFilters: [isnotnull(annee#10834), isnotnull(mois#10835), (annee#10834 = 2020), (mois#10835 = 11)]
ReadSchema: struct<horodatage:string,capacite:int,velos_meca:int,velos_elec:int,bornettes_libres:int,nom_station:string,coordonnees:string,operative:boolean,velos_total:int,taux_occupation:double>

(2) ColumnarToRow [codegen id : 1]
Input [12]: [horodatage#10824, capacite#10825, velos_meca#10826, velos_elec#10827, bornettes_libres#10828, nom_station#10829, coordonnees#10830, operative#10831, velos_t

---
## 2.3 Exploration et diagnostic des donnees

Avant tout traitement, il faut comprendre la structure et la qualite des donnees.
Spark propose des fonctions d'agregation statistiques integrees.


In [48]:
from pyspark.sql import functions as F

# Vue d'ensemble statistique
print("=== Statistiques descriptives (colonnes numeriques) ===")
cols_num = [
    "capacite", "velos_meca", "velos_elec",
    "bornettes_libres", "velos_total", "taux_occupation",
]
df_velib.select(*cols_num).describe().show()

=== Statistiques descriptives (colonnes numeriques) ===
+-------+------------------+------------------+------------------+------------------+------------------+--------------------+
|summary|          capacite|        velos_meca|        velos_elec|  bornettes_libres|       velos_total|     taux_occupation|
+-------+------------------+------------------+------------------+------------------+------------------+--------------------+
|  count|            690858|            690858|            690858|            690858|            690858|              690858|
|   mean| 33.40966739908925|0.8229448019708826|1.1748318757255471| 31.41189072139282|1.9977766776964296|  0.0603960472460285|
| stddev|11.426848004942414|0.9120225717753808|0.9065325106683281|10.828484360407373|1.0294126862459156|0.022011857418315756|
|    min|                11|                 0|                 0|                10|                 1|0.014285714285714285|
|    max|                70|                 6|               

In [49]:
from pyspark.sql.functions import col

# Comptage des valeurs nulles par colonne
print("=== Valeurs nulles par colonne ===")
for c in df_velib.columns:
    n_null = df_velib.filter(col(c).isNull()).count()
    print(f"  {c:<25} : {n_null:,}")

=== Valeurs nulles par colonne ===
  horodatage                : 0
  capacite                  : 0
  velos_meca                : 0
  velos_elec                : 0
  bornettes_libres          : 0
  nom_station               : 0
  coordonnees               : 0
  operative                 : 0
  velos_total               : 0
  taux_occupation           : 0
  annee                     : 0
  mois                      : 0


In [50]:
from pyspark.sql.functions import col

# Detection des anomalies evidentes
print("=== Anomalies detectees ===")

n_cap_zero = df_velib.filter(col("capacite") <= 0).count()
print(f"  Snapshots avec capacite <= 0          : {n_cap_zero:,}")

n_debord = df_velib.filter(col("velos_total") > col("capacite") + 2).count()
print(f"  Snapshots avec total > capacite + 2   : {n_debord:,}")

for col_name in ["velos_meca", "velos_elec", "bornettes_libres"]:
    n_neg = df_velib.filter(col(col_name) < 0).count()
    print(f"  {col_name:<30} < 0 : {n_neg:,}")


=== Anomalies detectees ===
  Snapshots avec capacite <= 0          : 0
  Snapshots avec total > capacite + 2   : 0
  velos_meca                     < 0 : 0
  velos_elec                     < 0 : 0
  bornettes_libres               < 0 : 0


In [51]:
# Distribution temporelle : combien de snapshots par mois ?
print("=== Couverture temporelle ===")
(
    df_velib
    .groupBy("annee", "mois")
    .count()
    .orderBy("annee", "mois")
    .show(truncate=False)
)


=== Couverture temporelle ===
+-----+----+------+
|annee|mois|count |
+-----+----+------+
|2020 |11  |69208 |
|2020 |12  |391383|
|2021 |1   |177367|
|2021 |2   |52900 |
+-----+----+------+



---
## 2.4 Nettoyage et construction des features

Nous allons maintenant construire la table `disponibilite` propre, telle que
definie dans le schema cible du projet.


In [52]:
from pyspark.sql.functions import (
    to_timestamp, col, when, round as spark_round,
    year, month, dayofweek, hour, greatest, lit,
)

# ── 1. Parsing et typage du timestamp ───────────────────────────────────────
df_clean = df_velib.withColumn(
    "ts", to_timestamp("horodatage", "yyyy-MM-dd'T'HH:mmX")
)

# ── 2. Suppression des lignes avec timestamp invalide ───────────────────────
df_clean = df_clean.filter(col("ts").isNotNull())

# ── 3. Suppression des stations avec capacite invalide ──────────────────────
df_clean = df_clean.filter(col("capacite") > 0)

# ── 4. Correction des valeurs negatives (bornage a 0) ───────────────────────
for c in ["velos_meca", "velos_elec", "bornettes_libres"]:
    df_clean = df_clean.withColumn(c, greatest(col(c), lit(0)))

# ── 5. Calcul du taux d'occupation ──────────────────────────────────────────
df_clean = df_clean.withColumn(
    "taux_occupation",
    spark_round((col("capacite") - col("bornettes_libres")) / col("capacite"), 4),
)

# ── 6. Bornage du taux entre 0 et 1 ─────────────────────────────────────────
df_clean = df_clean.withColumn(
    "taux_occupation",
    when(col("taux_occupation") < 0, 0.0)
    .when(col("taux_occupation") > 1, 1.0)
    .otherwise(col("taux_occupation")),
)

# ── 7. Extraction de features temporelles ───────────────────────────────────
df_clean = (
    df_clean
    .withColumn("annee", year("ts"))
    .withColumn("mois", month("ts"))
    .withColumn("jour_sem", dayofweek("ts"))
    .withColumn("heure", hour("ts"))
    .withColumn("est_weekend", col("jour_sem").isin(1, 7))  # dim=1, sam=7
)

print(f"Lignes apres nettoyage : {df_clean.count():,}")
df_clean.printSchema()
df_clean.show(3, truncate=True)


Lignes apres nettoyage : 690,858
root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = false)
 |-- velos_elec: integer (nullable = false)
 |-- bornettes_libres: integer (nullable = false)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- operative: boolean (nullable = true)
 |-- velos_total: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)

+-----------------+--------+----------+----------+----------------+--------------------+----------------+---------+-----------+---------------+-----+----+-------------------+--------+-----+-----------+
|       horodatage|capacite|velos_meca|velos_elec|bornettes_libres|         nom_stat

In [53]:
from pyspark.sql import functions as F

# [EXERCICE] colonne "statut"
df_clean = df_clean.withColumn(
    "statut",
    F.when(col("taux_occupation") < 0.1, "vide")
     .when(col("taux_occupation") > 0.9, "plein")
     .otherwise("normal"),
)

df_clean.groupBy("statut").count().orderBy("statut").show()


+------+------+
|statut| count|
+------+------+
|  vide|690858|
+------+------+



---
## 2.5 Chargement des donnees meteorologiques

La source meteorologique est le fichier SYNOP de Meteo-France pour la station
Paris-Montsouris, telechargeable librement sur data.gouv.fr.
Le format SYNOP est semi-structure et necessite quelques transformations.

Voici le format des colonnes pertinentes :

| Colonne SYNOP | Signification | Unite |
|---------------|---------------|-------|
| `AAAAMMJJHH`  | Horodatage    | UTC   |
| `T`           | Temperature   | K (Kelvin) |
| `U`           | Humidite relative | % |
| `FF`          | Vitesse du vent | m/s |
| `RR1`         | Precipitation | mm/h |
| `N`           | Nebulosity (couverture nuageuse) | octals (0-8) |


In [64]:
# Lecture du CSV meteo (format simplifie du projet, pas SYNOP brut)
# Colonnes : horodatage;temperature;precipitations;vent_ms;humidite;pression
df_meteo_raw = (
    spark.read
    .option("sep", ";")
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(METEO_CSV))
)

print(f"Lignes meteo brutes : {df_meteo_raw.count():,}")
df_meteo_raw.show(5)


Lignes meteo brutes : 35,064
+-------------------+-----------+--------------+-------+--------+--------+
|         horodatage|temperature|precipitations|vent_ms|humidite|pression|
+-------------------+-----------+--------------+-------+--------+--------+
|2020-01-01 00:00:00|        0.7|           0.0|   0.99|     100|  1022.8|
|2020-01-01 01:00:00|       -0.4|           0.0|   0.71|      99|  1023.2|
|2020-01-01 02:00:00|        2.4|           0.0|   1.03|      98|  1022.9|
|2020-01-01 03:00:00|        1.9|           0.0|   1.06|     100|  1022.5|
|2020-01-01 04:00:00|        1.7|           0.0|   1.36|     100|  1022.4|
+-------------------+-----------+--------------+-------+--------+--------+
only showing top 5 rows



In [65]:
from pyspark.sql.functions import to_timestamp, col, when

# Transformation : horodatage → timestamp, renommage des colonnes
df_meteo = (
    df_meteo_raw
    .withColumn("horodatage_meteo", to_timestamp("horodatage", "yyyy-MM-dd'T'HH:mm"))
    .withColumnRenamed("temperature", "temperature_c")
    .withColumnRenamed("humidite", "humidite_pct")
    .withColumn("vent_kmh", col("vent_ms") * 3.6)
    .withColumnRenamed("precipitations", "precipitation_mm")
    .filter(col("horodatage_meteo").isNotNull())
    .select("horodatage_meteo", "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm")
)

# Indicateur pluie
df_meteo = df_meteo.withColumn(
    "est_pluie",
    when(col("precipitation_mm") > 0, True).otherwise(False),
)

print(f"Lignes meteo nettoyees : {df_meteo.count():,}")
df_meteo.show(8)
df_meteo.describe(["temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"]).show()


Lignes meteo nettoyees : 35,064
+-------------------+-------------+------------+------------------+----------------+---------+
|   horodatage_meteo|temperature_c|humidite_pct|          vent_kmh|precipitation_mm|est_pluie|
+-------------------+-------------+------------+------------------+----------------+---------+
|2020-01-01 00:00:00|          0.7|         100|             3.564|             0.0|    false|
|2020-01-01 01:00:00|         -0.4|          99|             2.556|             0.0|    false|
|2020-01-01 02:00:00|          2.4|          98|             3.708|             0.0|    false|
|2020-01-01 03:00:00|          1.9|         100|3.8160000000000003|             0.0|    false|
|2020-01-01 04:00:00|          1.7|         100| 4.896000000000001|             0.0|    false|
|2020-01-01 05:00:00|          2.0|          98|             7.272|             0.0|    false|
|2020-01-01 06:00:00|          1.8|          99|             7.956|             0.0|    false|
|2020-01-01 07:00:

---
## 2.6 Jointure temporelle : Velib' x Meteo

Les observations Velib' sont a la minute, les donnees SYNOP a l'heure.
Pour joindre les deux tables, on **tronque** les horodatages Velib' a l'heure la plus
proche, puis on effectue une jointure sur cette cle commune.


In [66]:
from pyspark.sql.functions import date_trunc, broadcast, col

# Referentiel stations (station_id, code_arr) via nom
df_stations = (
    spark.read
    .option("sep", ";")
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STATIONS_CSV))
    .select(
        col("name").alias("nom_station"),
        col("station_id").cast("long"),
        col("code_arr").cast("int"),
    )
    .dropDuplicates(["nom_station"])
)

# ── 1. Cle de jointure : heure tronquee ──────────────────────────────────────
df_velib_join = df_clean.withColumn(
    "heure_tronquee",
    date_trunc("hour", col("ts")),
)

df_meteo_join = df_meteo.withColumnRenamed("horodatage_meteo", "heure_tronquee")

# ── 2. Broadcast join (meteo petite table) ────────────────────────────────────
df_joint = (
    df_velib_join
    .join(broadcast(df_meteo_join), on="heure_tronquee", how="left")
    .join(broadcast(df_stations), on="nom_station", how="left")
)

# Verification
n_avant  = df_clean.count()
n_apres  = df_joint.count()
n_meteo_null = df_joint.filter(col("temperature_c").isNull()).count()

print(f"Lignes avant jointure  : {n_avant:,}")
print(f"Lignes apres jointure  : {n_apres:,}")
print(f"Snapshots sans meteo   : {n_meteo_null:,} ({100*n_meteo_null/n_apres:.1f}%)")
print("(Velib 2020-2021 vs meteo 2022 → beaucoup de nulls est normal ici)")


Lignes avant jointure  : 690,858
Lignes apres jointure  : 690,858
Snapshots sans meteo   : 690,858 (100.0%)
(Velib 2020-2021 vs meteo 2022 → beaucoup de nulls est normal ici)


In [57]:
# Examinons le plan d'execution pour verifier que le broadcast est bien utilise
print("=== Plan d'execution de la jointure ===")
df_joint.explain(mode="formatted")
# Cherchez "BroadcastHashJoin" dans le plan -- pas de "SortMergeJoin" (qui implique un shuffle)


=== Plan d'execution de la jointure ===
== Physical Plan ==
AdaptiveSparkPlan (48)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- AdaptiveSparkPlan (47)
               +- == Final Plan ==
                  * Project (28)
                  +- * BroadcastHashJoin LeftOuter BuildRight (27)
                     :- * Project (16)
                     :  +- * BroadcastHashJoin LeftOuter BuildRight (15)
                     :     :- * Project (9)
                     :     :  +- * Project (8)
                     :     :     +- * Project (7)
                     :     :        +- * Project (6)
                     :     :           +- * Filter (5)
                     :     :              +- * ColumnarToRow (4)
                     :     :                 +- Scan parquet  (3)
                     :     +- BroadcastQueryStage (14), Statistics(sizeInBytes=9.0 MiB, rowCount=1.75E+4)
                     :        +- BroadcastExchange (13)
                     :           +-

In [67]:
from pyspark.sql import functions as F

# [EXERCICE] taux moyen par arrondissement et est_pluie
(
    df_joint
    .filter(col("code_arr").isNotNull())
    .groupBy("code_arr", "est_pluie")
    .agg(
        F.mean("taux_occupation").alias("taux_moyen"),
        F.count("*").alias("nb_snapshots"),
    )
    .orderBy("code_arr", "est_pluie")
    .show(20, truncate=False)
)


+--------+---------+--------------------+------------+
|code_arr|est_pluie|taux_moyen          |nb_snapshots|
+--------+---------+--------------------+------------+
|6       |NULL     |0.06205162880411474 |4666        |
|7       |NULL     |0.06596427889207282 |3141        |
|29      |NULL     |0.08285925925925926 |81          |
|30      |NULL     |0.06579172852598092 |943         |
|36      |NULL     |0.06753963270142187 |1688        |
|37      |NULL     |0.05702149712092128 |521         |
|38      |NULL     |0.06543450928381948 |3770        |
|52      |NULL     |0.05997484805984059 |6417        |
|63      |NULL     |0.0690039629005059  |1186        |
|77      |NULL     |0.06220218340611354 |229         |
|10968   |NULL     |0.05259999999999999 |454         |
|11375   |NULL     |0.062051282051282075|702         |
|11580   |NULL     |0.05879999999999999 |71          |
|13105   |NULL     |0.07105971404541622 |1189        |
|13191   |NULL     |0.06384642857142873 |1176        |
|13242   |

---
## 2.7 Persistance en memoire : `.cache()` et `.persist()`

Nous allons utiliser `df_joint` intensivement pendant le reste du cours.
Mettons-le en cache pour eviter de recalculer la jointure a chaque action.


In [ ]:
# .cache() == .persist(StorageLevel.MEMORY_AND_DISK)
# Si les donnees ne tiennent pas en RAM, Spark deborde sur disque.

t0 = time.perf_counter()
df_joint.cache()
df_joint.count()   # force la materialisation du cache
t_cache = time.perf_counter() - t0
print(f"Mise en cache (premiere passe) : {t_cache:.2f} s")

# Deuxieme passe -- depuis le cache
t0 = time.perf_counter()
df_joint.count()
t_hit = time.perf_counter() - t0
print(f"Lecture depuis le cache        : {t_hit:.2f} s")
print(f"Gain                           : x{t_cache/t_hit:.1f}")
print()
print("Allez dans Spark UI -> Storage pour voir la taille du cache.")


Mise en cache (premiere passe) : 0.05 s
Lecture depuis le cache        : 0.03 s
Gain                           : x1.9

Allez dans Spark UI -> Storage pour voir la taille du cache.


26/07/21 16:40:20 WARN CacheManager: Asked to cache already cached data.


26/07/21 20:24:06 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1038257 ms exceeds timeout 120000 ms
26/07/21 20:24:06 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/21 20:24:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

In [60]:
# StorageLevel disponibles (par ordre de rapidite / consommation memoire)
from pyspark import StorageLevel

print("StorageLevels disponibles :")
print("  MEMORY_ONLY          : RAM uniquement (eviction si plein)")
print("  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]")
print("  DISK_ONLY            : Disque uniquement (lent mais stable)")
print("  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)")
print("  OFF_HEAP             : Memoire hors JVM (necessite configuration)")
print()
print("Regle pratique :")
print("  - Petits DataFrames reutilises souvent -> MEMORY_ONLY")
print("  - Gros DataFrames reutilises -> MEMORY_AND_DISK")
print("  - Ne pas cacher si utilise une seule fois -> overhead inutile")


StorageLevels disponibles :
  MEMORY_ONLY          : RAM uniquement (eviction si plein)
  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]
  DISK_ONLY            : Disque uniquement (lent mais stable)
  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)
  OFF_HEAP             : Memoire hors JVM (necessite configuration)

Regle pratique :
  - Petits DataFrames reutilises souvent -> MEMORY_ONLY
  - Gros DataFrames reutilises -> MEMORY_AND_DISK
  - Ne pas cacher si utilise une seule fois -> overhead inutile


---
## 2.8 Ecriture en Parquet partitionne

La table `df_joint` est la table centrale du projet. Nous allons la persister
sur disque en Parquet, partitionne par annee et par mois, pour que les
traitements des jours suivants n'aient pas a la reconstruire.


In [61]:
OUTPUT_VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"

colonnes_finales = [
    "station_id", "nom_station", "code_arr", "capacite",
    "horodatage", "velos_meca", "velos_elec", "bornettes_libres",
    "taux_occupation", "statut",
    "annee", "mois", "jour_sem", "heure", "est_weekend",
    "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm", "est_pluie",
]

df_sortie = df_joint.select(*colonnes_finales)

t0 = time.perf_counter()
(
    df_sortie.write
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .parquet(str(OUTPUT_VELIB_CONSOLIDE))
)
t_write = time.perf_counter() - t0
print(f"Ecriture en {t_write:.1f} s")

df_verif = spark.read.parquet(str(OUTPUT_VELIB_CONSOLIDE)).filter("annee = 2020 AND mois = 11")
print(f"Novembre 2020 : {df_verif.count():,} lignes")
print(f"Schema : {df_verif.columns}")


Ecriture en 0.3 s
Novembre 2020 : 69,208 lignes
Schema : ['station_id', 'nom_station', 'code_arr', 'capacite', 'horodatage', 'velos_meca', 'velos_elec', 'bornettes_libres', 'taux_occupation', 'statut', 'jour_sem', 'heure', 'est_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'annee', 'mois']


In [62]:
# Comparaison de la taille CSV vs Parquet

def taille_dossier_mb(path: Path) -> float:
    """Calcule la taille d'un dossier en Mo."""
    if not path.exists():
        return 0.0
    if path.is_file():
        return path.stat().st_size / 1_048_576
    total = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return total / 1_048_576

t_csv  = taille_dossier_mb(HISTORIQUE_STATIONS_CSV)
t_parq = taille_dossier_mb(OUTPUT_VELIB_CONSOLIDE)
ratio  = t_csv / t_parq if t_parq > 0 else 0

print(f"Taille CSV historique              : {t_csv:.1f} MB")
print(f"Taille Parquet consolide          : {t_parq:.1f} MB")
print(f"Rapport CSV/Parquet               : x{ratio:.1f}")


Taille CSV historique              : 376.2 MB
Taille Parquet consolide          : 4.9 MB
Rapport CSV/Parquet               : x76.3


---
## Bilan du Jour 1

### Ce que nous avons fait

| Etape | API | Concept cle |
|-------|-----|-------------|
| Chargement CSV brut | RDD | `sc.textFile()`, parsing manuel |
| Filtrage et transformation | RDD | `map()`, `filter()`, evaluation paresseuse |
| Agregation par cle | RDD | `reduceByKey()` vs `groupByKey()` |
| Profil horaire | RDD | `flatMap()`, calcul de moyenne distribuee |
| Jointure RDD | RDD | `join()` sur paires (cle, valeur) |
| Lecture Parquet | DataFrame | `spark.read.parquet()`, predicate pushdown |
| Exploration statistique | DataFrame | `describe()`, comptage de nulls |
| Nettoyage | DataFrame | `dropna()`, `when()`, `greatest()` |
| Features temporelles | DataFrame | `year()`, `hour()`, `dayofweek()` |
| Jointure broadcast | DataFrame | `broadcast()`, eviter le shuffle |
| Persistance | DataFrame/RDD | `.cache()`, `.persist()`, `StorageLevel` |
| Ecriture partitionnee | DataFrame | `write.partitionBy().parquet()` |

### Concepts fondamentaux a retenir

1. **Evaluation paresseuse** : les transformations construisent un plan, les actions l'executent.
2. **Le DAG** : lire le Spark UI est une competence essentielle, pas optionnelle.
3. **reduceByKey > groupByKey** : toujours combiner localement avant de shuffler.
4. **DataFrame > RDD** : l'optimiseur Catalyst rend les DataFrames significativement plus
   rapides que les RDD pour les memes operations.
5. **Cache strategique** : ne mettre en cache que ce qui est reutilise plusieurs fois.
6. **Broadcast join** : quand une table est petite, eviter le shuffle de la grosse table.

### Pour demain (Jour 2)

La table `disponibilite_consolidee.parquet` sera le point de depart du Jour 2.
Nous allons l'interroger avec Spark SQL (fenetrage temporel, requetes analytiques),
puis la connecter a un flux simule de mises a jour en temps reel avec Structured Streaming.


In [63]:
# Nettoyage de fin de session
# Toujours liberer la SparkSession proprement pour liberer les ressources
print("Arret de la SparkSession...")
#spark.stop()
print("SparkSession arretee. Bonne nuit !")


Arret de la SparkSession...
SparkSession arretee. Bonne nuit !
